# Sequential curling final-score models

Configure and run the sequential experiment from `sequential_training.py`.

In [1]:
import importlib
import sys
import json
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / 'data_generation.py').exists():
    if repo_root.parent == repo_root:
        raise RuntimeError('Could not find repository root')
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import numpy as np
import data_generation
from scratch import sequential_training as seq
importlib.reload(seq)

N = 5
rows_per_model = 100_000
searcher = seq.make_default_throw_searcher(seed=0)
# artifact_candidates = [repo_root / 'scratch' / 'sequential_experiment', repo_root / 'scratch' / 'scratch' / 'sequential_experiment']
# artifact_dir = next((path for path in artifact_candidates if path.exists()), artifact_candidates[0])
# # Set to the next dataset to generate when resuming; use None for a fresh run.
# resume_from = 8

artifact_dir = repo_root / "scratch" / "sequential_experiment_100k"
resume_from = None

# Completed models listed here are retrained from their saved D_i datasets.
retrain_completed_models = {2 * N - 1} if resume_from is not None else set()
# Comparison-only searchers: the model uses the fixed grid, while greedy
# and the final throw use the same finer grid plus random candidates.
model_comparison_searcher = seq.ThrowsGridSearcher(10, 10, 4)
greedy_comparison_searcher = seq.GridAndRandomThrowSearcher(
    np.random.default_rng(0), grid_size=(10, 10, 4)
)

def print_policy_comparison(i):
    states = data_generation.random_sheet_states(
        team1=i // 2, team2=(i - 1) // 2, num_sims=100,
        rng=np.random.default_rng(i),
    )
    comparison = seq.compare_policies(
        i, states, models, max_stones=2 * N,
        searcher=greedy_comparison_searcher,
        model_searcher=model_comparison_searcher,
    )
    print(i, seq.policy_summary(comparison))
    seq.write_policy_comparison(
        artifact_dir / f'policy_comparison_{i}.npz', comparison
    )


In [2]:
models = {}
datasets = {}
validation_datasets = {}
training_info = {}
if resume_from is None:
    first_model_to_train = 2 * N - 1
else:
    first_model_to_train = resume_from
    for completed_i in range(resume_from + 1, 2 * N):
        if completed_i in retrain_completed_models:
            generated = seq.load_sequential_dataset(artifact_dir / f'D_{completed_i}.npz')
            datasets[completed_i] = generated
            data = generated.training_data()
            model, normalizer, info, validation_data = seq.train_model(data, seed=completed_i, max_stones=2 * N, num_stones_per_side=N)
            validation_datasets[completed_i] = validation_data
            models[completed_i] = (model, normalizer)
            training_info[completed_i] = info
            seq.write_model(artifact_dir / f'm_{completed_i}.npz', model, normalizer)
            seq.save_model_evaluation(artifact_dir / f'm_{completed_i}_evaluation.json', model=model, normalizer=normalizer, data=validation_data, training_info=info, model_index=completed_i, N=N)
            print_policy_comparison(completed_i)
        else:
            models[completed_i] = seq.load_model(artifact_dir / f'm_{completed_i}.npz')
            evaluation_path = artifact_dir / f'm_{completed_i}_evaluation.json'
            if evaluation_path.exists():
                training_info[completed_i] = json.loads(evaluation_path.read_text())
for i in range(first_model_to_train, 0, -1):
    generated = seq.generate_dataset(i, num_rows=rows_per_model, N=N, seed=i, models=models, searcher=searcher, shard_dir=artifact_dir / f'D_{i}', shard_size=500)
    datasets[i] = generated
    data = generated.training_data()
    model, normalizer, info, validation_data = seq.train_model(data, seed=i, max_stones=2 * N, num_stones_per_side=N)
    validation_datasets[i] = validation_data
    models[i] = (model, normalizer)
    training_info[i] = info
    seq.write_sequential_dataset(artifact_dir / f'D_{i}.npz', generated)
    seq.write_model(artifact_dir / f'm_{i}.npz', model, normalizer)
    seq.save_model_evaluation(artifact_dir / f'm_{i}_evaluation.json', model=model, normalizer=normalizer, data=validation_data, training_info=info, model_index=i, N=N)
    print_policy_comparison(i)
print(f'trained {len(models)} models')


9 {'model_expected_score': -1.51, 'greedy_expected_score': -1.82, 'expected_score_difference': 0.31, 'model_win_probability': 0.26, 'model_tie_probability': 0.59, 'model_loss_probability': 0.15, 'greedy_win_probability': 0.15, 'greedy_tie_probability': 0.59, 'greedy_loss_probability': 0.26}
8 {'model_expected_score': -1.8, 'greedy_expected_score': -2.04, 'expected_score_difference': 0.24, 'model_win_probability': 0.15, 'model_tie_probability': 0.52, 'model_loss_probability': 0.33, 'greedy_win_probability': 0.33, 'greedy_tie_probability': 0.52, 'greedy_loss_probability': 0.15}



KeyboardInterrupt



In [ ]:
# Standard validation statistics for each intermediate model.
for i, (model, normalizer) in models.items():
    print(i, seq.evaluate_model(model, normalizer, validation_datasets[i], N=N))


In [ ]:
# Policy comparisons are printed immediately after each model trains above.


In [ ]:
# Policy comparisons are printed immediately after each model trains above.


In [ ]:
# Final model data can use any requested mixture of stone counts.
final_data = seq.generate_final_dataset(
    models, k=1, fractions={i: 1 / (2 * N - 1) for i in range(1, 2 * N)},
    num_rows=rows_per_model, N=N, seed=123,
)
final_model, final_normalizer, final_info, final_validation_data = seq.train_model(
    final_data, seed=123, max_stones=2 * N, num_stones_per_side=N
)
seq.write_model(artifact_dir / 'final_model.npz', final_model, final_normalizer)
seq.save_metadata(artifact_dir / 'metadata.json', {
    'N': N, 'rows_per_model': rows_per_model, 'feature_width': seq.feature_width(2 * N),
    'training_info': training_info, 'final_training_info': final_info,
})
